# weight-decay-decoupled — ex2: compare one AdamW step against one coupled-Adam-with-L2 step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `weight-decay-decoupled`. Running the final beacon cell reports progress against the `Optimizer: decoupled weight decay (AdamW)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-decoupled`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-decoupled"
DD_SUBTOPIC = "Optimizer: decoupled weight decay (AdamW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Decoupled (AdamW) vs coupled (Adam+L2) — different θ after one step

Ex1 implemented one AdamW step where weight decay is applied DIRECTLY to θ, decoupled from the adaptive gradient:
```
# AdamW (decoupled):
m ← β1·m + (1−β1)·g
v ← β2·v + (1−β2)·g²
m̂ = m / (1 − β1^t)
v̂ = v / (1 − β2^t)
θ ← θ − lr·(m̂ / (√v̂ + ε)) − lr·wd·θ
```

Coupled Adam-with-L2 (the BUGGY pre-AdamW formulation) folds wd·θ into the gradient FIRST, then runs the adaptive Adam update on that augmented gradient:
```
# Coupled Adam + L2:
g' = g + wd·θ
m ← β1·m + (1−β1)·g'
v ← β2·v + (1−β2)·g'²
m̂ = m / (1 − β1^t)
v̂ = v / (1 − β2^t)
θ ← θ − lr·(m̂ / (√v̂ + ε))
```

**The two are NOT equivalent.** Because Adam normalises by `√v̂`, folding `wd·θ` into `g` makes the decay's effective magnitude depend on the second-moment estimate — small-gradient parameters get DECAYED LESS than they should. Decoupling fixes this. Two implementations on the SAME `g`, `m`, `v`, `θ` will produce DIFFERENT `θ_new`.

**Loshchilov & Hutter (2019)** showed this is why "Adam with weight decay" generalised worse than SGD+L2: the two updates the world thought were the same were not.

### Exercise 2 — compare one AdamW step against one coupled-Adam-with-L2 step

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyse the difference between decoupled (AdamW) and coupled (Adam+L2) weight-decay by implementing BOTH one-step updates on the same inputs and showing the resulting parameters differ — because Adam's `√v̂` normalisation re-scales decay only in the coupled path.
> Keywords: adamw, weight-decay, coupled, decoupled
> ```

**KCs targeted:** `decoupled-decay-direct-on-theta`, `coupled-decay-folded-into-gradient`

Implement `ex2_compare_decay_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step)`. Return a tuple `(p_adamw, p_coupled)` of the two updated parameters.

Both paths share inputs (`p`, `grad`, `m`, `v`, hparams, `step`). Treat `m`, `v` as immutable from the caller's perspective: do NOT mutate them — clone inside if needed.

**AdamW path (decoupled):**
```
m_aw ← β1·m + (1−β1)·grad
v_aw ← β2·v + (1−β2)·grad²
m̂_aw = m_aw / (1 − β1**step)
v̂_aw = v_aw / (1 − β2**step)
p_adamw = p − lr · (m̂_aw / (√v̂_aw + eps))  −  lr · wd · p
```

**Coupled path (Adam + L2):**
```
g' = grad + wd · p
m_cp ← β1·m + (1−β1)·g'
v_cp ← β2·v + (1−β2)·g'²
m̂_cp = m_cp / (1 − β1**step)
v̂_cp = v_cp / (1 − β2**step)
p_coupled = p − lr · (m̂_cp / (√v̂_cp + eps))
```

Inputs: `p`, `grad`, `m`, `v` are all `Tensor`s of the same shape. `step` is 1-indexed (matches PyTorch's Adam state).

Output: tuple `(p_adamw, p_coupled)` — both fresh tensors. Original `p` is unchanged.

In [ ]:
def ex2_compare_decay_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    # --- AdamW path: decoupled decay applied to p directly ---
    m_aw = beta1 * m + (1 - beta1) * grad
    v_aw = beta2 * v + (1 - beta2) * grad * grad
    m_hat_aw = m_aw / (1 - beta1 ** step)
    v_hat_aw = v_aw / (1 - beta2 ** step)
    p_adamw = p - lr * (m_hat_aw / (v_hat_aw.sqrt() + eps)) - lr * wd * p

    # --- Coupled path: fold wd·p into gradient, then run plain Adam ---
    g_eff = grad + wd * p
    m_cp = beta1 * m + (1 - beta1) * g_eff
    v_cp = beta2 * v + (1 - beta2) * g_eff * g_eff
    m_hat_cp = m_cp / (1 - beta1 ** step)
    v_hat_cp = v_cp / (1 - beta2 ** step)
    p_coupled = p - lr * (m_hat_cp / (v_hat_cp.sqrt() + eps))

    return p_adamw, p_coupled


<details><summary>Solution</summary>

```python
def ex2_compare_decay_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    # --- AdamW path: decoupled decay applied to p directly ---
    m_aw = beta1 * m + (1 - beta1) * grad
    v_aw = beta2 * v + (1 - beta2) * grad * grad
    m_hat_aw = m_aw / (1 - beta1 ** step)
    v_hat_aw = v_aw / (1 - beta2 ** step)
    p_adamw = p - lr * (m_hat_aw / (v_hat_aw.sqrt() + eps)) - lr * wd * p

    # --- Coupled path: fold wd·p into gradient, then run plain Adam ---
    g_eff = grad + wd * p
    m_cp = beta1 * m + (1 - beta1) * g_eff
    v_cp = beta2 * v + (1 - beta2) * g_eff * g_eff
    m_hat_cp = m_cp / (1 - beta1 ** step)
    v_hat_cp = v_cp / (1 - beta2 ** step)
    p_coupled = p - lr * (m_hat_cp / (v_hat_cp.sqrt() + eps))

    return p_adamw, p_coupled
```

**Why the wd=0 case must coincide.** When `wd=0`, the AdamW decay term vanishes; the coupled path's augmented gradient `g + 0·p = g` is the same as plain Adam. Both updates collapse to vanilla Adam. This is the boundary test — if your implementation disagrees here, you have a bug in the BASE update, not in the decay handling.

**Why `v.sqrt() + eps` not `(v + eps).sqrt()`.** PyTorch's Adam uses the former (`addcdiv_` form). For non-pathological `v̂`, both round to the same value. PyTorch tests against the official Adam paper's formulation; matching it makes regression tests cleaner.

**The scalar handwork.** Step 1, scalar p, m=v=0: the bias-corrected first/second moment perfectly cancel each other's normalisation (`m̂/√v̂ = sign(g)`), so the adaptive step is exactly 1. This makes the WHOLE update reducible to mental arithmetic — a useful sanity check pattern for any Adam variant.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()